# **Demo de predicción CPU y GPU**

Se carga un modelo entrenado y se muestran predicciones con los kernels CUDA.
La comparación final usa las mismas ocho imágenes de validación y los mismos
pesos en CPU y GPU. Seleccioná un entorno con GPU NVIDIA para ejecutar el cuaderno.


## **Preparar el repositorio**

Se ejecuta al iniciar el notebook. Si la carpeta existe, se actualizan las
referencias; si no existe, se clona. Se muestra el commit utilizado. Si cambian
modulos Python ya importados, se reinicia la sesion antes de continuar.


In [ ]:
from pathlib import Path
import subprocess

url_repositorio = "https://github.com/Bamo0507/Proyecto_3_Cuda_CNN.git"
revision_repositorio = "origin/main"
directorio_proyecto = Path("/content/Proyecto_3_Cuda_CNN")

if directorio_proyecto.exists():
    subprocess.run(["git", "fetch", "origin"], cwd=directorio_proyecto, check=True)
else:
    subprocess.run(["git", "clone", url_repositorio, str(directorio_proyecto)], check=True)

subprocess.run(
    ["git", "checkout", "--detach", revision_repositorio],
    cwd=directorio_proyecto,
    check=True,
)
commit_ejecutado = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=directorio_proyecto, text=True
).strip()
print("Commit ejecutado:", commit_ejecutado)


## **Instalar dependencias CUDA**

Se instalan las dependencias de Colab y Matplotlib. Si se solicita reiniciar la
sesion, hacerlo antes de continuar y ejecutar nuevamente la preparacion del repositorio.


In [ ]:
import subprocess
import sys

resultado_instalacion = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     str(directorio_proyecto / "requirements_colab.txt"), "matplotlib"],
    capture_output=True,
    text=True,
)
print(resultado_instalacion.stdout, end="")
if resultado_instalacion.stderr:
    print(resultado_instalacion.stderr, end="")
resultado_instalacion.check_returncode()


## **Montar Drive para acceder a los tensores**

La carpeta proyecto_senias debe aparecer en Mi unidad y contener 02_tensores.npz.
Los pesos y su registro se cargan desde el repositorio.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

directorio_drive = Path("/content/drive/MyDrive/proyecto_senias")
nombre_ejecucion = "dani_gpu_01"


## **Cargar el modelo y las imágenes de validación**

Se verifica que el orden de las clases y la forma de entrada correspondan al
modelo. Se leen las imágenes y las etiquetas de la partición de validación.


In [ ]:
import json
import sys
import numpy as np

sys.path.insert(0, str(directorio_proyecto / "src"))

import config
from compartido.persistencia import cargar_parametros
from numba import cuda
from gpu.memoria import preparar_memoria
from gpu.prediccion import predecir

ruta_modelo_local = directorio_proyecto / "models" / "dani" / f"{nombre_ejecucion}.npz"
ruta_registro_local = (
    directorio_proyecto / "resultados" / "dani" / f"entrenamiento_{nombre_ejecucion}.json"
)
ruta_tensores_drive = directorio_drive / "02_tensores.npz"
if not ruta_tensores_drive.is_file():
    raise FileNotFoundError(f"No se encontró el archivo de tensores en {ruta_tensores_drive}")

registro_modelo = json.loads(ruta_registro_local.read_text())
if registro_modelo["configuracion"]["clases"] != list(config.CLASES):
    raise ValueError("El orden de clases del modelo no coincide con config.py.")
if registro_modelo["configuracion"]["forma_entrada"] != list(config.FORMA_ENTRADA):
    raise ValueError("La forma de entrada del modelo no coincide con config.py.")

parametros = cargar_parametros(ruta_modelo_local)
with np.load(ruta_tensores_drive, allow_pickle=False) as archivo_tensores:
    if not np.array_equal(archivo_tensores["clases"], np.asarray(config.CLASES)):
        raise ValueError("El orden de clases del dataset no coincide con el modelo.")
    imagenes_validacion = archivo_tensores["x_val"]
    etiquetas_validacion = archivo_tensores["y_val"]

if etiquetas_validacion.shape != (len(imagenes_validacion),):
    raise ValueError("Debe existir una etiqueta por imagen de validacion.")
print("Modelo cargado:", nombre_ejecucion)
print("Imagenes de validacion:", len(imagenes_validacion))


## **Preparar memoria y realizar la primera prediccion**

Se copian los pesos a GPU y se reservan buffers una sola vez. La primera prediccion
puede incluir compilacion y se registra por separado. La copia del resultado a RAM
completa los kernels pendientes antes de devolver la etiqueta.


In [ ]:
import time

if cuda.config.ENABLE_CUDASIM or not cuda.is_available():
    raise RuntimeError("Selecciona una GPU NVIDIA real antes de continuar.")
print("GPU:", cuda.get_current_device().name)

inicio_preparacion = time.perf_counter()
memoria_prediccion = preparar_memoria(
    parametros, tamano_lote=config.TAMANO_LOTE, entrenamiento=False
)
cuda.synchronize()
tiempo_preparacion = time.perf_counter() - inicio_preparacion

indice_imagen = 0
imagen_individual = imagenes_validacion[indice_imagen:indice_imagen + 1]
cuda.synchronize()
inicio_calentamiento = time.perf_counter()
prediccion_individual = predecir(imagen_individual, memoria_prediccion)[0]
tiempo_primera_prediccion = time.perf_counter() - inicio_calentamiento

print("Etiqueta real:", config.CLASES[int(etiquetas_validacion[indice_imagen])])
print("Prediccion:", prediccion_individual["etiqueta"])
print(f"Probabilidad: {prediccion_individual['probabilidad']:.2%}")
print(f"Preparacion de memoria y pesos: {tiempo_preparacion:.4f} s")
print(f"Primera prediccion y posible compilacion: {tiempo_primera_prediccion:.4f} s")


## **Revisar ejemplos de validacion**

Se seleccionan cinco ejemplos con semilla fija y se muestran sus etiquetas reales
y predichas. El color permite distinguir aciertos y errores; estos ejemplos no
sustituyen la accuracy calculada sobre toda la validacion.


In [ ]:
import matplotlib.pyplot as graficos

cantidad_ejemplos = min(5, len(imagenes_validacion))
generador_aleatorio = np.random.default_rng(config.SEMILLA_ALEATORIA)
indices_ejemplos = generador_aleatorio.choice(
    len(imagenes_validacion), size=cantidad_ejemplos, replace=False
)
predicciones_ejemplos = predecir(imagenes_validacion[indices_ejemplos], memoria_prediccion)

figura, ejes = graficos.subplots(
    1, cantidad_ejemplos, figsize=(3 * cantidad_ejemplos, 3), squeeze=False
)
for indice_ejemplo, indice_validacion in enumerate(indices_ejemplos):
    prediccion = predicciones_ejemplos[indice_ejemplo]
    etiqueta_real = config.CLASES[int(etiquetas_validacion[indice_validacion])]
    eje_imagen = ejes[0, indice_ejemplo]
    eje_imagen.imshow(imagenes_validacion[indice_validacion, :, :, 0], cmap="gray", vmin=0, vmax=1)
    eje_imagen.set_title(
        f"Real: {etiqueta_real}\nPred: {prediccion['etiqueta']} ({prediccion['probabilidad']:.1%})",
        color="green" if prediccion["etiqueta"] == etiqueta_real else "red",
    )
    eje_imagen.axis("off")
graficos.tight_layout()
graficos.show()


## **Medir la prediccion despues del calentamiento**

Se repite 30 veces la prediccion de una imagen. Cada tiempo incluye transferencia
de la imagen RAM a GPU, forward, recuperacion de probabilidades y seleccion de
etiqueta en CPU. Los pesos y buffers permanecen en GPU. Se excluyen Drive, carga
del modelo, reserva inicial y compilacion.

Esta comprobacion de latencia no sustituye los experimentos de Daniela. Para
comparar backends, utilizar los mismos pesos, entradas y condiciones.


In [ ]:
cantidad_repeticiones = 30
tiempos_prediccion = []

for numero_repeticion in range(cantidad_repeticiones):
    cuda.synchronize()
    inicio_prediccion = time.perf_counter()
    prediccion_medida = predecir(imagen_individual, memoria_prediccion)[0]
    tiempos_prediccion.append(time.perf_counter() - inicio_prediccion)

tiempos_milisegundos = np.asarray(tiempos_prediccion) * 1000
print("Repeticiones:", cantidad_repeticiones)
print(f"Latencia promedio: {tiempos_milisegundos.mean():.4f} ms")
print(f"Desviacion estandar: {tiempos_milisegundos.std(ddof=1):.4f} ms")
print("Tiempos individuales (ms):", tiempos_milisegundos.tolist())


## **Comparar CPU y GPU con los mismos pesos**

Se evalúan ocho imágenes de validación. La primera llamada CPU puede incluir
compilación de Numba. Esta muestra sirve para demostrar el funcionamiento.


In [ ]:
from cpu.prediccion import evaluar as evaluar_cpu
from gpu.prediccion import evaluar as evaluar_gpu

cantidad_demo = 8
imagenes_demo = imagenes_validacion[:cantidad_demo]
etiquetas_demo = etiquetas_validacion[:cantidad_demo]

print("CPU:", evaluar_cpu(imagenes_demo, etiquetas_demo, parametros))
print("GPU:", evaluar_gpu(imagenes_demo, etiquetas_demo, memoria_prediccion))
